# Figure 4 - Visium analysis

# Load Modules

In [ ]:
from pathlib import Path
import sys
import importlib
import glob
import anndata as an
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import squidpy as sq
from matplotlib.patches import Patch


import os

import seaborn as sns
import scipy.stats as stats
import scikit_posthocs as sp

plt.rcdefaults()

code_dir = "/projects/Pregnancy_placenta/Code_Publication/Python/"
HELPER_FUNCS_DIR = Path(code_dir) / "Functions"
TMA_PATH = Path(
    "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/data/Final_Objs/final_TMA_obj_V3.h5ad"
)
FINAL_FIG_DIR = Path(
    "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/results/final_figures/Figure_4"
)

SCALE_BAR_UM = 100
SCALE_BAR_BBOX = (0.97, -0.018)
LINE_SIZE_VERTICAL = 1.2


## load functions 

In [ ]:
if str(HELPER_FUNCS_DIR) not in sys.path:
    sys.path.append(str(HELPER_FUNCS_DIR))

import vis_tcell_helpful_func as vis_tcr_plt
importlib.reload(vis_tcr_plt)

import visium_plotting as vis_plt
importlib.reload(vis_plt)   # if you edit it and want to refresh

import ipa_pathway_analysis as ipa_analysis
importlib.reload(ipa_analysis)

vis_plt.configure_publication_fonts()

# load obj 

In [ ]:
TMA_obj = an.read_h5ad(TMA_PATH)



# Figure 4D - Zoomined in Villi 

In [ ]:
import matplotlib.gridspec as gridspec

mpl.rcParams.update(
    {
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

mask = (
    (TMA_obj.obs["Type_of_Villi"] == "Terminal")
    & (TMA_obj.obs["Concerns"] == "N")
)
sub_TMA = TMA_obj[mask].copy()

crop = (1180, 1640, 1280, 1740)
library_id = "TMA2-140"
scale_bar_px = vis_plt.get_scale_bar_px(
    sub_TMA[sub_TMA.obs["library_id"] == library_id].copy(),
    scale_bar_um=SCALE_BAR_UM,
)

fig = plt.figure(figsize=(7.2, 3.6))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.02)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

vis_plt.plot_tma_crop_or_rect(
    sub_TMA,
    zoom_rect=crop,
    img=True,
    size=1,
    shape="square",
    library_id=library_id,
    orientation="bottom_right",
    title=None,
    ax=ax1,
)
vis_plt.add_scale_bar(
    ax1,
    scale_bar_px,
    label=f"{SCALE_BAR_UM} µm",
    bbox_to_anchor=(0.97, -0.01),
    bbox_transform=ax1.transAxes,
    size_vertical=2,
    zorder=30,
)
ax1.set_title("H&E with Terminal Villi Zoom", pad=6, fontweight="bold")
ax1.set_xticks([])
ax1.set_yticks([])
ax1.set_xlabel("")
ax1.set_ylabel("")

sq.pl.spatial_scatter(
    sub_TMA,
    color=["dominant_parent_type"],
    img=True,
    library_key="library_id",
    library_id=library_id,
    shape="square",
    size=1,
    crop_coord=crop,
    legend_fontsize=13,
    ax=ax2,
)
vis_plt.add_scale_bar(
    ax2,
    scale_bar_px,
    bbox_to_anchor=SCALE_BAR_BBOX,
    bbox_transform=ax2.transAxes,
    size_vertical=LINE_SIZE_VERTICAL,
    zorder=30,
)
ax2.set_title(
    "Highest Proportion of\nMain Lineage (Zoomed Terminal Villi)",
    pad=6,
    fontweight="bold",
)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_xlabel("")
ax2.set_ylabel("")

plt.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.05)

vis_plt.save_figure(
    fig,
    "Fig4D_TMA2-140_bottom_right_VC_PTBV2.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.show()

# Figure 4E - Violin plot 

## helper functions 

In [ ]:

def p_to_star(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "n.s"

def add_bracket(ax, x1, x2, y, label):
    ax.plot([x1, x1, x2, x2], [y, y, y, y], color="black", lw=2, clip_on=False)
    ax.text(
        (x1 + x2) / 2,
        y + 0.004,
        label,
        ha="center",
        va="bottom",
        fontsize=13,
        fontweight="bold"
    )


## prep villi 

In [ ]:

villus_col = "which_VC"
type_col = "Type_of_Villi"
conc_col = "Concerns"
cond_col = "Condition"
patient_col = "Patient"

hbc_col = "ab_parent__HBC"
total_col = "total_cell_abundance"  # you already computed this earlier

# Filter Bins
obs_filt = TMA_obj.obs.copy()

obs_filt = obs_filt[
    (obs_filt[villus_col].notna()) &
    (obs_filt[villus_col] != "Not_in_VC") &
    (obs_filt[type_col] == "Terminal") &
    (obs_filt[conc_col] == "N")
    ].copy()

obs_filt = obs_filt[obs_filt[total_col] > 0].copy()

### aggregate bisn into villi level 
villi_df = (
    obs_filt
    .groupby(villus_col, observed=True)
    .agg(
        n_bins_v=(villus_col, "size"),
        HBC_sum_v=(hbc_col, "sum"),
        Total_sum_v=(total_col, "sum"),
        Patient=(patient_col, "first"),
        Condition=(cond_col, "first"),
        Type_of_Villi=(type_col, "first"),
        Concerns=(conc_col, "first"),
    )
    .reset_index()
)

# derived outcomes
villi_df["HBC_density_v"] = villi_df["HBC_sum_v"] / villi_df["n_bins_v"]
villi_df["HBC_prop_v"] = villi_df["HBC_sum_v"] / (villi_df["Total_sum_v"])


# After villi_df is created
if pd.api.types.is_categorical_dtype(villi_df["Condition"]):
    villi_df["Condition"] = villi_df["Condition"].cat.remove_unused_categories()


In [ ]:

os.chdir(FINAL_FIG_DIR)

mpl.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

cond_order = ["Control (Labored)", "Control (Unlabored)", "Preterm Birth"]
xtick_labels = ["Labored\ncontrol", "Unlabored\ncontrol", "sPTB"]

palette = {
    "Control (Labored)": "#eeeeee",
    "Control (Unlabored)": "#eeeeee",
    "Preterm Birth": "#bfbfbf"
}

dfp = villi_df.copy()
dfp["Condition"] = pd.Categorical(
    dfp["Condition"],
    categories=cond_order,
    ordered=True
)

# Stats
groups = [dfp.loc[dfp["Condition"] == c, "HBC_prop_v"].dropna() for c in cond_order]
kw_stat, kw_p = stats.kruskal(*groups)
print(f"Kruskal-Wallis p = {kw_p:.3g}")

dunn = sp.posthoc_dunn(
    dfp,
    val_col="HBC_prop_v",
    group_col="Condition",
    p_adjust="bonferroni"
)

# PLOT
fig, ax = plt.subplots(figsize=(6.1, 4.2))

sns.violinplot(
    data=dfp,
    x="Condition",
    y="HBC_prop_v",
    order=cond_order,
    palette=palette,
    cut=0,
    inner="quartile",
    linewidth=3,
    ax=ax
)

sns.stripplot(
    data=dfp,
    x="Condition",
    y="HBC_prop_v",
    order=cond_order,
    color="#6b6b6b",
    size=4,
    jitter=0.25,
    alpha=0.75,
    ax=ax
)

# Style violins and quartile lines
for coll in ax.collections:
    coll.set_edgecolor("#6b6b6b")
    coll.set_linewidth(2.5)

for line in ax.lines:
    line.set_color("#6b6b6b")
    line.set_linewidth(2.5)

# ADD BRACKETS
pairs = [
    ("Control (Labored)", "Control (Unlabored)"),
    ("Control (Labored)", "Preterm Birth"),
    ("Control (Unlabored)", "Preterm Birth"),
]

xpos = {c: i for i, c in enumerate(cond_order)}

y_start = dfp["HBC_prop_v"].max() + 0.015
y_step = 0.032

for i, (c1, c2) in enumerate(pairs):
    p = dunn.loc[c1, c2]
    add_bracket(
        ax,
        xpos[c1],
        xpos[c2],
        y_start + i * y_step,
        p_to_star(p)
    )

# STYLE LABELS
ax.set_xlabel("")
ax.set_ylabel("HBC / Total cells per villi", fontweight="bold")
ax.set_xticklabels(xtick_labels, fontweight="bold")
ax.set_title("")

ax.set_ylim(
    dfp["HBC_prop_v"].min() - 0.01,
    y_start + len(pairs) * y_step + 0.015
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(2)

ax.tick_params(axis="both", width=2, length=5)

plt.subplots_adjust(left=0.12, right=0.98, top=0.96, bottom=0.22)


fig.savefig(
    "Fig4E_HBC_proportion_violin_manuscript_V1.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Sup Table - Angio Score + Mixed MOdel

## subset to Terminal villi 

In [ ]:
mask = (
        (TMA_obj.obs['Type_of_Villi'] == "Terminal") &
        (TMA_obj.obs['Concerns'] == "N")
)

sub_TMA = TMA_obj[mask].copy()
sub_TMA


## Prepare IPA genes used to score 

### load genes 

In [ ]:
dir_to_pathway = "/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/Code_worksapce/IPA_results"
all_files = [os.path.basename(p) for p in glob.glob(os.path.join(dir_to_pathway, "*"))]
ipa_merged_df = ipa_analysis.read_and_merge_IPA_outputs(dir_to_pathway, recursive=False, skiprows=1, verbose=False)
meta_cols = ["cell_type", "condition", "control", "annotation_type"]
other_cols = [c for c in ipa_merged_df.columns if c not in meta_cols]

ipa_merged_df = ipa_merged_df[meta_cols + other_cols]


### Subset to just pathway of interest 

In [ ]:
path_of_interest = "Angiogenesis"
sub_ipa_df = ipa_merged_df.loc[
    ipa_merged_df['Diseases or Functions Annotation'] ==path_of_interest
].copy()

sub_ipa_df['unique_group'] = (
    sub_ipa_df['cell_type']
    + "-"
    + sub_ipa_df['condition']
    + "-"
    + sub_ipa_df['control']
)

sub_ipa_df['IPA_group'] = (
    sub_ipa_df['cell_type']
    + "-"
    + sub_ipa_df['condition']
    + "_vs_"
    + sub_ipa_df['control']
)

## elimiante Cont_PTB samples
sub_ipa_df = sub_ipa_df[~sub_ipa_df['control'].isin(['Cont_PTB'])].copy()

## get angiogenesis gene lo2fc values 

### load log2fc values 

In [ ]:
angio_log2fc = pd.read_csv("/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/Code_worksapce/IPA_results/log2fc_genes/angiogenesis_genes_HBC_PTB_all_log2fc_vals.csv",index_col = "gene_name")

In [ ]:
ipa_cols = sub_ipa_df["IPA_group"].unique()

df_angio = angio_log2fc[ipa_cols].copy()

df_angio = df_angio.dropna( how="all", subset=df_angio.columns.difference(["gene_name"]) )

### filter #1 - eliminate neg log2fc genes 

In [ ]:
df_angio["keep_for_score"] = np.where(
    df_angio[df_angio.columns.difference(["gene_name"])].lt(0).any(axis=1),
    "N",
    "Y"
)

# add reason for filtering column
df_angio["Reason_for_filtering"] = ""

df_angio.loc[
    df_angio["keep_for_score"] == "N",
    "Reason_for_filtering"
] = "Negative Fold Change"

### Filter #2: eliminate low variance genes 

In [ ]:
pos_genes = df_angio.index[df_angio["keep_for_score"] == "Y"].tolist()


In [ ]:
df_filter = vis_plt.pathway_filter_table(
    sub_TMA,
    pos_genes,
    min_frac=0.03,
    min_mean=0
)

### add info of filter#2 step to df_angio

In [ ]:
# Columns that are actual log2FC comparison columns
fc_cols = df_angio.columns.difference([
    "keep_for_score",
    "Reason_for_filtering",
    "mean_expr",
    "frac_nonzero"
])

# TRUE negative fold-change genes based on actual values
negative_fc_mask = df_angio[fc_cols].lt(0).any(axis=1)

# Rebuild filtering columns cleanly
df_angio["keep_for_score"] = "Y"
df_angio["Reason_for_filtering"] = ""

# First: mark actual negative fold-change genes
df_angio.loc[negative_fc_mask, "keep_for_score"] = "N"
df_angio.loc[negative_fc_mask, "Reason_for_filtering"] = "Negative Fold Change"

# Set up df_filter by gene
df_filter_idx = df_filter.set_index("gene")

# Add expression stats where available
df_angio["mean_expr"] = df_angio.index.map(df_filter_idx["mean_expr"])
df_angio["frac_nonzero"] = df_angio.index.map(df_filter_idx["frac_nonzero"])

# For actual negative fold-change genes, expression stats were not calculated/used
df_angio.loc[negative_fc_mask, ["mean_expr", "frac_nonzero"]] = "Not Calculated - Negative"

# Now mark genes that failed the df_filter step
failed_filter_genes = df_filter_idx.index[df_filter_idx["keep"] == False]
failed_mask = df_angio.index.isin(failed_filter_genes)

# Only apply df_filter reasons to genes that were NOT already negative FC
failed_nonnegative_mask = failed_mask & ~negative_fc_mask

df_angio.loc[failed_nonnegative_mask, "keep_for_score"] = "N"
df_angio.loc[failed_nonnegative_mask, "Reason_for_filtering"] = (
    df_angio.loc[failed_nonnegative_mask].index.map(df_filter_idx["reason"])
)
path_score_genes =df_angio[df_angio['keep_for_score']=="Y"].index.tolist()


### score pathway 

In [ ]:
genes_pass_qc = df_filter.loc[df_filter["keep"], "gene"].tolist()

vis_plt.signature_score_per_cell(
    data=sub_TMA,          # or whatever AnnData you’re scoring
    gene_set=path_score_genes,
    score_title="Vasculogenesis_pos_genes_filt_mean_frac")

## Sup Table 13 - angiogenesis score 

In [ ]:
df_angio.to_csv("/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/results/Final_Supp_tables/Supp_Table_13_AngioGenesis_Score.csv")

## Linear Mixed Model Analysis 

### assign variables 

In [ ]:
signature_col = "Vasculogenesis_pos_genes_filt_mean_frac"
condition_order = ["Preterm Birth","Control (Labored)", "Control (Unlabored)"]


In [ ]:

obs_used = vis_plt.prep_bin_covariates_filtered(
    sub_TMA,
    signature_col=signature_col,
    condition_col="Condition",
    patient_col="Patient",
    villus_col="which_VC",
    villus_exclude_value="Not_in_VC",
    villi_type_col="Type_of_Villi",
    villi_type_keep="Terminal",
    hbc_col="ab_parent__HBC",
    total_abund_col="total_cell_abundance",
    condition_order=condition_order,
)

### run models 

In [ ]:
## full
r_main_ml = vis_plt.fit_signature_mixedlm_FULL(
    obs_used,
    signature_col=signature_col,
    reml=False,
)

## reduced
r_red_ml = vis_plt.fit_signature_mixedlm_REDUCED(
    obs_used,
    signature_col=signature_col,
    reml=False,
)

## LRT
from scipy import stats
import statsmodels.formula.api as smf

lr_stat_cond = 2 * (r_main_ml.llf - r_red_ml.llf)
df_cond = r_main_ml.df_modelwc - r_red_ml.df_modelwc
pval_cond = stats.chi2.sf(lr_stat_cond, df_cond)



## save as CSV

In [ ]:
# Adjusting Variables
output_csv = "Supplementary_Table_Spatial_HBC_Vasculogenesis_MixedLM.csv"

reference_condition = "Preterm Birth"

n_bins = obs_used.shape[0]
n_patients = obs_used["Patient"].nunique()
n_villi = (
    obs_used["patient_villus"].nunique()
    if "patient_villus" in obs_used.columns
    else obs_used["which_VC"].nunique()
)

rows = []

models = {
    "Reduced model: Condition + HBC_frac": r_red_ml,
    "Full model: Condition × HBC_frac": r_main_ml,
}

# fixed effect variables from models
for model_name, res in models.items():
    ci = res.conf_int()

    for term in res.fe_params.index:
        coef = res.fe_params.loc[term]
        se = res.bse_fe.loc[term]
        pval = res.pvalues.loc[term]
        ci_low = ci.loc[term, 0]
        ci_high = ci.loc[term, 1]

        rows.append({
            "model": model_name,
            "reference_condition": reference_condition,
            "term": term,
            "estimate": coef,
            "std_error": se,
            "z_value": coef / se,
            "p_value": pval,
            "ci_95_low": ci_low,
            "ci_95_high": ci_high,
            "n_bins": n_bins,
            "n_patients": n_patients,
            "n_villi": n_villi,
            "log_likelihood": res.llf,
        })

# add in LRT
rows.append({
    "model": "Likelihood ratio test: full vs reduced model",
    "reference_condition": reference_condition,
    "term": "Condition × HBC_frac interaction",
    "estimate": lr_stat_cond,
    "std_error": None,
    "z_value": None,
    "p_value": pval_cond,
    "ci_95_low": None,
    "ci_95_high": None,
    "n_bins": n_bins,
    "n_patients": n_patients,
    "n_villi": n_villi,
    "log_likelihood": r_main_ml.llf,
    "lrt_df": df_cond,
    "reduced_model_log_likelihood": r_red_ml.llf,
    "full_model_log_likelihood": r_main_ml.llf,
})

# SAVE
supp_mixedlm_df = pd.DataFrame(rows)
supp_mixedlm_df.to_csv("/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/results/Final_Supp_tables/Supp_Table_14_mixed_model_analysis.csv",index=False)


# Figure 4F - Angiogenesis Plot

In [ ]:
# need to change from above to make it top left 
SCALE_BAR_BBOX = (0.03, 0.97)

## Row 1

In [ ]:
library_id = "TMA2-140"
crop = (1145, 1650, 1245, 1750)
scale_bar_px = vis_plt.get_scale_bar_px(
    sub_TMA[sub_TMA.obs["library_id"] == library_id].copy(),
    scale_bar_um=SCALE_BAR_UM,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))
fig.patch.set_facecolor("white")

sq.pl.spatial_scatter(
    sub_TMA,
    color=["HBC"],
    use_raw=False,
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    img=True,
    library_key="library_id",
    library_id=library_id,
    shape="square",
    size=1,
    outline=False,
    crop_coord=crop,
    title=None,
    ax=ax1,
)
vis_plt.add_scale_bar(
    ax1,
    scale_bar_px,
    loc="upper left",
    bbox_to_anchor=SCALE_BAR_BBOX,
    bbox_transform=ax1.transAxes,
    size_vertical=LINE_SIZE_VERTICAL,
    zorder=30,
)
vis_plt.clean_spatial_axis(
    ax1,
    "HBC Proportion per Bin",
    fontsize=12,
    pad=6,
    fontweight="bold",
)

sq.pl.spatial_scatter(
    sub_TMA,
    color=["Vasculogenesis_pos_genes_filt_mean_frac"],
    use_raw=False,
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    img=True,
    library_key="library_id",
    library_id=library_id,
    shape="square",
    size=1,
    outline=False,
    crop_coord=crop,
    title=None,
    ax=ax2,
)
vis_plt.add_scale_bar(
    ax2,
    scale_bar_px,
    loc="upper left",
    bbox_to_anchor=SCALE_BAR_BBOX,
    bbox_transform=ax2.transAxes,
    size_vertical=LINE_SIZE_VERTICAL,
    zorder=30,
)
vis_plt.clean_spatial_axis(
    ax2,
    "Angiogenesis score",
    fontsize=12,
    pad=6,
    fontweight="bold",
)

plt.subplots_adjust(left=0.02, right=0.98, bottom=0.06, top=0.9, wspace=0.04)

vis_plt.save_figure(
    fig,
    "Fig4F_top_angiogenesis_HBC_spatial.crop_TMA2-140_V2.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.show()

## Row 2

In [ ]:
library_id = "TMA2-145"
crop = (600, 1100, 700, 1200)
scale_bar_px = vis_plt.get_scale_bar_px(
    sub_TMA[sub_TMA.obs["library_id"] == library_id].copy(),
    scale_bar_um=SCALE_BAR_UM,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))
fig.patch.set_facecolor("white")

sq.pl.spatial_scatter(
    sub_TMA,
    color=["HBC"],
    use_raw=False,
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    img=True,
    library_key="library_id",
    library_id=library_id,
    shape="square",
    size=1,
    outline=False,
    crop_coord=crop,
    title=None,
    ax=ax1,
)
vis_plt.add_scale_bar(
    ax1,
    scale_bar_px,
    loc="upper left",
    bbox_to_anchor=SCALE_BAR_BBOX,
    bbox_transform=ax1.transAxes,
    size_vertical=LINE_SIZE_VERTICAL,
    zorder=30,
)
ax1.set_title("HBC Proportion per Bin", fontweight="bold", fontsize=12, pad=6)
vis_plt.clean_spatial_axis(
    ax1,
    "HBC Proportion per Bin",
    fontsize=12,
    pad=6,
    fontweight="bold",
)

sq.pl.spatial_scatter(
    sub_TMA,
    color=["Vasculogenesis_pos_genes_filt_mean_frac"],
    use_raw=False,
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    img=True,
    library_key="library_id",
    library_id=library_id,
    shape="square",
    size=1,
    outline=False,
    crop_coord=crop,
    title=None,
    ax=ax2,
)
vis_plt.add_scale_bar(
    ax2,
    scale_bar_px,
    loc="upper left",
    bbox_to_anchor=SCALE_BAR_BBOX,
    bbox_transform=ax2.transAxes,
    size_vertical=LINE_SIZE_VERTICAL,
    zorder=30,
)
ax2.set_title("Angiogenesis score", fontweight="bold", fontsize=12, pad=6)
vis_plt.clean_spatial_axis(
    ax2,
    "Angiogenesis score",
    fontsize=12,
    pad=6,
    fontweight="bold",
)

plt.subplots_adjust(left=0.02, right=0.98, bottom=0.06, top=0.9, wspace=0.04)

vis_plt.save_figure(
    fig,
    "Fig4F_bottom_angiogenesis_HBC_spatial.crop_TMA2-145_V2.pdf",
    FINAL_FIG_DIR,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01,
)
plt.show()